# 02 — Sync metadata: per-run summaries and watermarks

The sync writes one summary doc per (run, pipeline) to `sync_metadata`. Use it
to:

- Confirm the last run finished as `success` (vs. `partial` / `failure`).
- Track watermark progression for incremental pipelines (`learners`).
- Spot pipelines where `rows_failed` is non-zero.


In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
from azure.cosmos import CosmosClient

repo_root = Path.cwd()
while not (repo_root / ".env.example").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("could not find repo root (.env.example missing)")
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

endpoint = os.environ["COSMOS_ENDPOINT"]
key = os.environ.get("COSMOS_EMULATOR_KEY")  # emulator-only escape hatch
db_name = os.environ.get("COSMOS_DATABASE", "learnsphere")

if key:
    client = CosmosClient(endpoint, credential=key)
else:
    from azure.identity import DefaultAzureCredential
    client = CosmosClient(endpoint, credential=DefaultAzureCredential())

db = client.get_database_client(db_name)
print(f"endpoint={endpoint}  database={db_name}  auth={'key' if key else 'aad'}")


## Last 10 runs


In [ ]:
import pandas as pd
container = db.get_container_client("sync_metadata")
items = list(container.query_items(
    """
    SELECT TOP 10 c.run_id, c.pipeline, c.status, c.rows_read,
           c.rows_upserted, c.rows_failed, c.duration_ms, c.watermark, c.finished_at
    FROM c
    ORDER BY c.finished_at DESC
    """,
    enable_cross_partition_query=True,
))
pd.DataFrame(items)


## Watermark progression for the `learners` pipeline


In [ ]:
import pandas as pd
items = list(container.query_items(
    """
    SELECT c.run_id, c.watermark, c.rows_upserted, c.finished_at
    FROM c
    WHERE c.pipeline = 'learners'
    ORDER BY c.finished_at DESC
    """,
    enable_cross_partition_query=True,
))
pd.DataFrame(items)
